# Análisis de contractilidad de geles 3D — flujo paso a paso (v3)

**Autores**

&nbsp;&nbsp;&nbsp;&nbsp;Franco Rita y Sofia Zavaleta

**Descripción**

&nbsp;&nbsp;&nbsp;&nbsp;Este cuaderno recorre, paso a paso y mostrando la salida de cada etapa, el flujo
automatizado que convierte un vídeo de microscopía de un gel 3D en una lista de contracciones
cuantificadas: lectura del vídeo, mapa de máxima intensidad, detección automática de la región
útil de medición (*gauge region*), localización subpíxel de los bordes columna a columna, ajuste
robusto RANSAC que descarta columnas corrompidas, construcción de las series temporales,
**elección del observable**, detección de eventos con verificación estadística del umbral,
medida del adelgazamiento por promedio de eventos alineados, y análisis del ritmo.

---

### Qué cambió respecto de la versión anterior de este cuaderno

La detección de bordes **no cambió**: sigue siendo el mismo subpíxel parabólico con ajuste
RANSAC. Lo que cambió es qué se hace con esos bordes.

1. **Se guarda la posición de cada borde, no sólo su resta.** Antes el cuaderno sólo producía
   `thickness_px` = borde inferior − borde superior. Ahora también `y_top_px`, `y_bottom_px` y
   `center_px` (el promedio de los dos). Motivo: si los dos bordes se mueven **juntos**, la resta
   no cambia y el grosor es ciego a ese movimiento; el promedio no lo es.
2. **El observable de detección se elige con datos, no por costumbre** (sección 7). Medido sobre
   los vídeos del proyecto, la contracción de este montaje es mayormente un desplazamiento
   vertical de toda la franja, y sólo un 13–19 % de ese movimiento es cambio de grosor. Detectar
   sobre `thickness_px` es trabajar con un observable atenuado ~6×.
3. **El grosor se sigue midiendo, pero por promedio de eventos alineados** (sección 11), que es
   lo que permite verlo aunque por fotograma esté enterrado en el ruido.
4. **Parámetros de ajuste corregidos**: `ransac_degree` pasa de 1 a 2 y el umbral de residuo pasa
   de 1.5 px fijo a adaptativo. Con los valores viejos, ~70 % de la varianza de la serie de
   grosor era artefacto del propio ajuste.
5. **La *gauge region* se calcula con un criterio de planitud** y reporta las alternativas
   descartadas, en vez de un único intento que podía quedarse con el 20 % de la imagen.

**Paquetes necesarios**

&nbsp;&nbsp;&nbsp;&nbsp;`numpy`, `pandas`, `scipy`, `opencv-python`, `scikit-learn`, `matplotlib`, `openpyxl`

**Requisitos de archivos**

&nbsp;&nbsp;&nbsp;&nbsp;Un vídeo del gel (`.mp4` / `.avi`). El cuaderno se ejecuta desde la raíz del
proyecto (`gel_contractility/`), donde está `main.py`. No hace falta ImageJ.

---

### Mapa del flujo

| Paso | Qué entra | Qué sale |
|---|---|---|
| 0. Entorno | — | módulos cargados, rutas resueltas |
| 1. Parámetros | criterio del operador | configuración única para todo el cuaderno |
| 2. Lectura del vídeo | `.mp4` / `.avi` | metadatos + mapa de máxima intensidad |
| 3. Región útil (*gauge region*) | mapa de máxima intensidad | rango `x_start`–`x_end`, bordes aproximados, alternativas descartadas |
| 4. Preprocesado y borde subpíxel | un fotograma + región útil | fotograma normalizado + perfil de una columna |
| 5. Ajuste robusto (RANSAC) | ~60 bordes crudos | *overlay* inlier/outlier + grosor y posición de ese fotograma |
| 6. Series temporales | vídeo completo | tabla con 4 canales por fotograma |
| **7. Elección del observable** | los 4 canales | cuál tiene la señal: ruido, asimetría y SNR de cada uno |
| 8. Línea base, ruido y umbral | canal elegido | estado relajado, ruido medido, umbral |
| 9. Detección de contracciones | profundidad + umbral | tabla de eventos |
| 10. Verificación del umbral | serie temporal | meseta + control de falsos positivos |
| **11. Adelgazamiento y poblaciones** | eventos + grosor | promedio de eventos alineados, cociente adelgazamiento/traslación, grupos de eventos |
| 12. Ritmo | eventos | tramos de frecuencia homogénea + perfil de frecuencia |
| 13. Exportación | todo lo anterior | `serie_temporal.xlsx`, `eventos.xlsx`, `contracciones.xlsx` y los PNG numerados |

---

## 0. Preparación del entorno

Se cargan los módulos del proyecto (`src/`) y, con `importlib`, los dos scripts de `scripts/`
que no son importables como paquete. **No se reimplementa nada**: este cuaderno llama exactamente
a las mismas funciones que `main.py` y `scripts/contraction_report.py`, sólo que deteniéndose en
cada paso para mostrar el resultado intermedio.

In [ ]:
import sys, warnings, importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Raíz del proyecto: la carpeta que contiene main.py y src/
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import io_utils, preprocessing, edge_detection, robust_fitting, pipeline
from src import qc_visualization as qc
from src import event_detection as ed
from src import plotting                      # fija el backend Agg para guardar PNG
from src.output_paths import video_output_dir


def _load_script(nombre):
    # Carga un archivo de scripts/ como módulo (no son un paquete importable).
    spec = importlib.util.spec_from_file_location(nombre, PROJECT_ROOT / "scripts" / f"{nombre}.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

cr = _load_script("contraction_report")   # detección sobre center_px, promedio alineado
ac = _load_script("analyze_contractions") # sólo para _append_sheet en la exportación

warnings.filterwarnings("ignore")
print("Raíz del proyecto:", PROJECT_ROOT.name + "/")

## 1. Parámetros del análisis

**Todo lo ajustable vive en esta celda.** El resto del cuaderno no contiene ningún número mágico.

Asegurarse de fijar correctamente:

- **`VIDEO`** — ruta del vídeo a analizar.
- **`PX_TO_MM`** — factor de calibración. Se obtiene fotografiando una regla o retícula con el
  mismo objetivo y zoom con el que se graban los geles: `PX_TO_MM = mm_conocidos / píxeles_medidos`.
  Si se deja en `1.0`, **todos los resultados quedan en píxeles** y el cuaderno lo señala en los
  títulos de las figuras.
- **`half_window`** — semiancho (px) de la ventana donde se busca el borde en cada fotograma.
  Debe cubrir el rango máximo de deformación esperado en Y.
- **`min_gradient`** — gradiente mínimo para aceptar un borde. Se calibra mirando la sección 4.
  **Ojo:** se compara contra el gradiente de la imagen *ya pasada por CLAHE*, así que su valor no
  es portable entre vídeos si se cambia `use_clahe`.
- **`ransac_degree`** — grado del polinomio que modela el borde. **2**, no 1: el gel real tiene
  curvatura, y con una recta esa curvatura se clasifica como *outlier* y contamina la medición.
- **`ransac_residual_threshold`** — `None` = adaptativo (3×MAD de los residuos del propio
  fotograma). Fijarlo a mano sólo si se sabe exactamente por qué.
- **`CANAL`** — observable sobre el que se detectan los eventos. `"auto"` deja que la sección 7
  lo elija por SNR. Ponerlo a mano sólo para forzar una comparación.
- **`AMP_K`** — umbral de detección en múltiplos del ruido. **No fijarlo a dedo**: elegirlo con
  la curva de estabilidad de la sección 10.

In [ ]:
%matplotlib inline

# --- Entrada ---------------------------------------------------------------
VIDEO      = PROJECT_ROOT / "data" / "raw_videos" / "Video_prueba.mp4"
PX_TO_MM   = 1.0     # <-- calibrar. 1.0 = resultados en píxeles (SIN CALIBRAR)

# --- Mapa de máxima intensidad --------------------------------------------
MAXPROJ_STRIDE = 5   # procesa 1 de cada N fotogramas; 1 si el vídeo es corto

# --- Detección de bordes, ajuste robusto y gauge region --------------------
CFG = pipeline.PipelineConfig(
    n_columns                 = 60,
    half_window               = 15,
    min_gradient              = 5.0,
    edge_method               = "parabolic",   # "parabolic" | "sigmoid"

    fit_method                = "ransac",      # "ransac" | "median"
    ransac_degree             = 2,             # 2, no 1: el borde real tiene curvatura
    ransac_residual_threshold = None,          # None = adaptativo (3 x MAD del fotograma)
    ransac_residual_k         = 3.0,
    ransac_residual_floor     = 0.4,

    roi_x_start               = None,          # None = automático; forzar sólo si hace falta
    roi_x_end                 = None,
    roi_thickness_tolerance   = 0.05,
    roi_min_gradient          = 10.0,
    roi_max_slope             = 0.02,

    px_to_mm                  = PX_TO_MM,
    use_clahe                 = True,
    use_denoise               = False,
    savgol_window             = 11,
    savgol_polyorder          = 3,
    low_quality_frac          = 0.30,
)

# --- Elección del observable y detección de eventos ------------------------
CANAL         = "auto"   # "auto" | "center_px" | "thickness_px" | "y_top_px" | "y_bottom_px"
AMP_K         = 6.0      # umbral = AMP_K * ruido  (verificar con la sección 10)
SHARPNESS     = 1.30     # agudeza mínima; 0 la desactiva
MIN_AMP_PX    = None     # piso absoluto de amplitud (opcional)
NOISE_PX      = None     # ruido medido en un vídeo de control (opcional)
WIN_DERIVA_S  = 2.0      # ventana de la mediana móvil que quita la deriva (secciones 7 y 11)
HALF_EVENTO_S = 1.5      # semiventana del promedio de eventos alineados (sección 11)
FREQ_WINDOW_S = 8.0      # ventana del perfil de frecuencia (sección 12)

# --- Control de calidad -----------------------------------------------------
FRAME_QC = 0             # fotograma sobre el que se hace la inspección visual

# --- Salida -----------------------------------------------------------------
CALIBRADO = abs(PX_TO_MM - 1.0) > 1e-9
UNIDAD    = "mm" if CALIBRADO else "px (SIN CALIBRAR)"
OUT_DIR   = video_output_dir(VIDEO)
print("Vídeo :", VIDEO.name)
print("Salida:", OUT_DIR.relative_to(PROJECT_ROOT))
print("Unidad:", UNIDAD)

## 2. Lectura del vídeo y mapa de máxima intensidad

**Entra:** el vídeo crudo. **Sale:** sus metadatos y una única imagen resumen.

El vídeo se lee fotograma a fotograma con un generador (`io_utils.frame_generator`), nunca entero
en memoria: en vídeos largos de microscopía eso es la diferencia entre correr el análisis o
quedarse sin RAM.

El **mapa de máxima intensidad** guarda, para cada píxel, el valor máximo que alcanzó a lo largo
del vídeo. Es el equivalente exacto de *Z-Project > Max Intensity* de ImageJ. Sirve para saber
**dónde puede estar el borde del gel en cualquier instante** sin indicarlo a mano: la zona que se
movió en algún momento queda iluminada.

**Ejes de la figura:** `x` = columna de píxeles de la imagen (0 a ancho−1, izquierda a derecha);
`y` = fila de píxeles (0 arriba, creciente hacia abajo — convención de imagen, no de gráfico).
El gris es intensidad máxima acumulada, 0 negro a 255 blanco.

In [ ]:
meta = io_utils.get_video_metadata(str(VIDEO))
fps  = meta["fps"] if meta["fps"] > 0 else 30.0
meta["duracion_s"] = round(meta["n_frames"] / fps, 2)
display(pd.DataFrame([meta]).T.rename(columns={0: "valor"}))

max_proj = io_utils.compute_max_projection(str(VIDEO), stride=MAXPROJ_STRIDE)

fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(max_proj, cmap="gray")
ax.set_title(f"Mapa de máxima intensidad — {VIDEO.name} (1 de cada {MAXPROJ_STRIDE} fotogramas)")
ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
plt.show()

## 3. Región útil de medición (*gauge region*)

**Entra:** el mapa de máxima intensidad. **Sale:** el rango de columnas donde se va a medir y la
posición aproximada de los bordes superior e inferior.

El gel no es un rectángulo: tiene forma de reloj de arena, delgado y uniforme en el centro y más
ancho cerca de los anclajes. Medir junto a los anclajes es incorrecto desde lo experimental (ahí
manda la concentración de tensiones del anclaje, no la contractilidad del material).

`preprocessing.auto_detect_roi` **sigue la franja columna a columna desde el centro hacia afuera**
(así un halo o un reflejo no la estiran), mide grosor y nitidez de cada columna, estima la
**cintura** del gel usando sólo columnas cuyo grosor es compatible con el gel, y busca el bloque
contiguo más largo que cumpla, en cascada de menos a más permisivo:

| nivel | criterio |
|---|---|
| `gauge_plana` | nítida + cerca de la cintura + perfil plano |
| `gauge_cintura` | nítida + cerca de la cintura |
| `gauge_relajada` | nítida + tolerancia de grosor al doble |
| `solo_nitidez` | sólo nitidez |
| `franja_completa` | todo lo que se pudo seguir |

Se queda con el primer nivel que dé un bloque suficientemente ancho, y **reporta también los que
descartó**, para poder forzar otro a mano con `roi_x_start` / `roi_x_end` si conviene.

**Ejes de `00_roi_profile.png`:** panel superior — `x` = columna de la imagen (px), `y` = grosor
de la franja en esa columna (px); la línea punteada es la cintura estimada y la banda verde la
ROI elegida. Panel inferior — mismo `x`, `y` = nitidez del borde, definida como el **mínimo** del
gradiente vertical máximo de los dos bordes (mínimo, no máximo: los dos tienen que ser nítidos).

In [ ]:
roi = preprocessing.auto_detect_roi(
    max_proj,
    thickness_tolerance  = CFG.roi_thickness_tolerance,
    min_gradient_for_roi = CFG.roi_min_gradient,
    max_thickness_slope  = CFG.roi_max_slope,
    x_start              = CFG.roi_x_start,
    x_end                = CFG.roi_x_end,
)
x_positions = np.linspace(roi["x_start"], roi["x_end"] - 1, CFG.n_columns).astype(int)

q = roi["roi_quality"]
display(pd.DataFrame([{k: v for k, v in q.items() if k != "alternativas"}]).T
        .rename(columns={0: "valor"}))

if q.get("alternativas"):
    print("Alternativas de ROI consideradas (la elegida es la primera suficientemente ancha):")
    display(pd.DataFrame(q["alternativas"]))

print(f"Columnas muestreadas: {CFG.n_columns} entre x={roi['x_start']} y x={roi['x_end']} "
      f"(1 cada {(roi['x_end']-roi['x_start'])/CFG.n_columns:.1f} px)")

# Perfil de grosor y nitidez con la ROI marcada
qc.plot_roi_profile(roi, OUT_DIR / "00_roi_profile.png")
display(Image(str(OUT_DIR / "00_roi_profile.png")))

# La misma ROI, sobre la imagen
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(max_proj, cmap="gray")
xs = np.arange(max_proj.shape[1])
ax.plot(xs, roi["top_guess"],    color="#56C48F", lw=1, label="borde superior aprox.")
ax.plot(xs, roi["bottom_guess"], color="#56C48F", lw=1)
ax.axvspan(roi["x_start"], roi["x_end"], color="#FFCA7F", alpha=0.25, label="gauge region")
ax.vlines(x_positions, roi["top_guess"][x_positions] - 20,
          roi["bottom_guess"][x_positions] + 20, color="#FFCA7F", lw=0.4, alpha=0.8)
ax.set_title("Región útil de medición y columnas muestreadas")
ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)"); ax.legend(loc="upper right", fontsize=8)
plt.show()

## 4. Preprocesado y localización subpíxel del borde

**Entra:** un fotograma y la región útil. **Sale:** el fotograma normalizado y el perfil de
intensidad/gradiente de una columna, con el borde localizado.

CLAHE normaliza el contraste **localmente** (por celdas), lo que compensa el viñeteo y las
variaciones de iluminación entre fotogramas. Comprobado sobre Video_063: con CLAHE el ruido de
`center_px` baja de 0.0495 a 0.0338 px (1.5× mejor) y la amplitud medida de los eventos cambia
sólo un 2 % — o sea que **mejora el SNR sin deformar la señal**.

Sobre cada columna se calcula el gradiente de intensidad y se ajusta una parábola a los tres
puntos alrededor de su máximo: el vértice de esa parábola es la posición del borde **con
resolución de fracciones de píxel**. Si el gradiente máximo no supera `min_gradient`, la columna
se descarta en vez de forzar una medición dudosa.

**Ejes de la figura de perfil:** `x` = fila `y` de la imagen (px) dentro de la ventana de
búsqueda; una curva es la intensidad del píxel (0–255) y la otra el gradiente vertical
(unidades de intensidad por píxel). La vertical marca el vértice de la parábola, es decir el
borde subpíxel.

In [ ]:
idx0, frame0 = next(io_utils.frame_generator(str(VIDEO), start_frame=FRAME_QC))
frame_p = preprocessing.preprocess_frame(frame0, use_clahe=CFG.use_clahe,
                                         use_denoise=CFG.use_denoise)

x0, x1 = roi["x_start"], roi["x_end"]
y0 = int(max(0, roi["top_guess"][x0:x1].min() - 40))
y1 = int(min(frame0.shape[0], roi["bottom_guess"][x0:x1].max() + 40))

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].imshow(frame0[y0:y1, x0:x1], cmap="gray"); axes[0].set_title(f"Fotograma {idx0} — crudo")
axes[1].imshow(frame_p[y0:y1, x0:x1], cmap="gray")
axes[1].set_title("Tras CLAHE (lo que analiza el algoritmo)")
for a in axes: a.set_yticks([])
plt.tight_layout(); plt.show()

# Perfil de UNA columna: intensidad, gradiente y borde subpíxel
x_col = int(np.median(x_positions))
fig = qc.plot_column_profile(frame_p, x_col, roi["top_guess"][x_col],
                             half_window=CFG.half_window, polarity=+1,
                             min_gradient=CFG.min_gradient)
plt.show()

pt = edge_detection.subpixel_edge_parabolic(
    frame_p[:, x_col].astype(float),
    int(roi["top_guess"][x_col] - CFG.half_window),
    int(roi["top_guess"][x_col] + CFG.half_window),
    polarity=+1, min_gradient=CFG.min_gradient)
print(f"Columna x={x_col}: borde superior en y={pt.y:.3f} px "
      f"(gradiente={pt.quality:.2f}, válido={pt.valid})")

## 5. Ajuste robusto: qué columnas se aceptan y cuáles se descartan

**Entra:** los ~60 bordes crudos del fotograma. **Sale:** el modelo ajustado, la lista de columnas
descartadas, y el grosor **y la posición** de ese fotograma.

Con 60 columnas es esperable que alguna esté tapada por una burbuja. Promediarlas todas diluiría
ese error. RANSAC ajusta el modelo a subconjuntos aleatorios, se queda con el que reúne más
*inliers* y **excluye** el resto por completo.

Dos cambios importantes respecto de la versión anterior:

- **`ransac_degree = 2`.** Con grado 1 (recta), la curvatura real del gel se clasificaba como
  outlier: en un fotograma limpio aparecían 9 outliers **contiguos** en el borde superior. Los
  outliers contiguos indican que el modelo está mal, los dispersos indican burbujas.
- **Umbral adaptativo.** En vez de 1.5 px fijo, se usa 3× el MAD de los residuos del propio
  fotograma. Con el umbral fijo, ~70 % de la varianza de la serie de grosor era artefacto del
  ajuste, no movimiento del gel.

En el *overlay*: verde = borde aceptado, rojo = borde detectado pero descartado como outlier,
amarillo = columna sin borde confiable, cian = modelo final (lo que realmente define el grosor).

In [ ]:
diag = qc.run_frame_diagnostics(frame_p, x_positions,
                                roi["top_guess"], roi["bottom_guess"], CFG)
overlay = qc.draw_diagnostics_overlay(frame_p, diag, px_to_mm=PX_TO_MM)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.imshow(overlay[y0:y1, x0:x1, ::-1])
ax.set_title(f"Fotograma {idx0} — verde: inlier · rojo: outlier · amarillo: sin borde · cian: ajuste")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

tabla = qc.diagnostics_to_dataframe(diag)
n = len(tabla)
print(f"Grosor del fotograma {idx0}: {diag.thickness_px:.3f} px"
      + (f"  ({diag.thickness_px * PX_TO_MM:.4f} mm)" if CALIBRADO else ""))
print(f"Columnas: {n} | borde superior outlier: {int((~tabla.top_is_inlier).sum())}"
      f" | borde inferior outlier: {int((~tabla.bottom_is_inlier).sum())}"
      f" | sin borde: {int(tabla.y_top_raw.isna().sum() + tabla.y_bottom_raw.isna().sum())}")

# Outliers CONTIGUOS = el modelo está mal.  DISPERSOS = burbujas.
for lado, col in (("superior", "top_is_inlier"), ("inferior", "bottom_is_inlier")):
    malos = np.flatnonzero(~tabla[col].to_numpy())
    if len(malos) >= 3:
        bloques = np.split(malos, np.flatnonzero(np.diff(malos) > 1) + 1)
        mayor = max(len(b) for b in bloques)
        diagn = ("CONTIGUOS -> el modelo no representa la geometría: subir ransac_degree"
                 if mayor >= 3 else "dispersos -> compatible con burbujas/motas")
        print(f"  borde {lado}: {len(malos)} outliers, bloque mayor de {mayor} -> {diagn}")

display(tabla.head(8).round(3))

## 6. Series temporales (vídeo completo)

**Entra:** el vídeo entero. **Sale:** una fila por fotograma, con **cuatro canales** en vez de uno.

Se repite el ciclo de las secciones 4 y 5 en cada fotograma. Cada uno queda etiquetado como `OK`,
`LOW_QUALITY` (más de `low_quality_frac` de las columnas descartadas, contando los dos bordes) o
`REJECTED` (ningún ajuste confiable); los `REJECTED` quedan como `NaN`, no se inventan.

Los cuatro canales, y qué ve cada uno:

| columna | qué es | ciego a |
|---|---|---|
| `y_top_px` | posición del borde superior (px, y crece hacia abajo) | — |
| `y_bottom_px` | posición del borde inferior | — |
| `thickness_px` | `y_bottom − y_top` | **traslación**: si los dos bordes bajan 1 px, no cambia |
| `center_px` | `(y_bottom + y_top) / 2` | **cambio de grosor**: si el borde de arriba baja 1 px y el de abajo sube 1 px, no cambia |

`thickness_px` y `center_px` son **complementarios**: entre los dos describen todo el movimiento
vertical de la franja. Guardar sólo uno es tirar la mitad de la información, que es exactamente lo
que hacía la versión anterior de este cuaderno.

Además se guardan `residual_top_px` y `residual_bottom_px` (MAD de los residuos del ajuste, por
fotograma). **Si esa columna se mueve junto con el grosor, lo que se está midiendo es el ajuste,
no el gel** — es el control interno del método.

Al final se aplica un suavizado temporal **Savitzky-Golay** en lugar de una media móvil: la media
móvil aplanaría los picos de contracción, mientras que Savitzky-Golay ajusta un polinomio local y
reduce el ruido **preservando la forma** de los eventos rápidos.

> Esta celda recorre todo el vídeo; es la más lenta del cuaderno.

In [ ]:
import cv2
OUT_DIR.mkdir(parents=True, exist_ok=True)
maxproj_path = OUT_DIR / "00_max_projection.png"
cv2.imwrite(str(maxproj_path), max_proj)

df = pipeline.process_video(str(VIDEO), str(maxproj_path), CFG)
t  = df.time_s.to_numpy()

resumen = {
    "fotogramas totales"     : len(df),
    "OK"                     : int((df.frame_quality == "OK").sum()),
    "LOW_QUALITY"            : int((df.frame_quality == "LOW_QUALITY").sum()),
    "REJECTED"               : int((df.frame_quality == "REJECTED").sum()),
    "grosor medio (px)"      : round(float(df.thickness_px.mean()), 3),
    "recorrido grosor (px)"  : round(float(df.thickness_px.max() - df.thickness_px.min()), 3),
    "recorrido centro (px)"  : round(float(df.center_px.max() - df.center_px.min()), 3),
    "outlier_frac medio"     : round(float(df.outlier_frac.mean()), 4),
    "residuo sup medio (px)" : round(float(df.residual_top_px.mean()), 4),
    "residuo inf medio (px)" : round(float(df.residual_bottom_px.mean()), 4),
}
display(pd.DataFrame([resumen]).T.rename(columns={0: "valor"}))
display(df.head(5).round(4))

if df.outlier_frac.mean() > 0.10:
    print("AVISO: se descarta más del 10 % de las columnas en promedio. Unos pocos por ciento es "
          "normal con burbujas; 10 % sostenido, no. Revisar la sección 5.")

# Los cuatro canales, uno debajo del otro y en la misma escala de tiempo
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, c, col in zip(axes, ["thickness_px", "center_px", "y_top_px", "y_bottom_px"],
                      ["crimson", "steelblue", "darkgreen", "darkorange"]):
    ax.plot(t, df[c], color=col, lw=0.7)
    ax.set_ylabel(f"{c}\n({UNIDAD})", fontsize=8)
    ax.grid(alpha=0.3)
axes[0].set_title("Los cuatro canales — mismo vídeo, misma escala de tiempo, distinta información")
axes[-1].set_xlabel("Tiempo (s)")
plt.tight_layout(); plt.show()

display(Image(str(plotting.plot_timeseries(df, OUT_DIR, unit_label=UNIDAD, calibrated=CALIBRADO))))

## 7. Elección del observable — ¿en qué canal está la contracción?

**Entra:** los cuatro canales. **Sale:** cuál de ellos se usa para detectar, y por qué.

Ésta es la sección nueva, y es la que cambia el resultado del análisis.

La intuición de que una contracción tiene que verse como **adelgazamiento** supone que los dos
extremos del gel están perfectamente fijos y que la contracción es simétrica. En la práctica, en
este montaje, la mayor parte del movimiento es un **desplazamiento vertical de toda la franja**, y
sólo un 13–19 % de él es cambio de grosor. Ese cociente resultó ser **el mismo en el vídeo que
"funcionaba" y en el que "no funcionaba"**: la mecánica es la misma, lo que cambia es la amplitud.

Para decidir sin suponer nada, se comparan los cuatro canales con tres números, todos calculados
después de quitar la deriva lenta con una **mediana móvil** (mediana, no pasabanda: un pasabanda
convierte cada evento en un valle flanqueado por dos picos falsos y destruye la asimetría que
justamente se quiere medir):

- **`ruido_MAD_px`** — mediana de las desviaciones absolutas, escalada para que sea comparable a
  un desvío estándar. Es el piso de ruido del canal.
- **`skew`** — asimetría. Una contracción es una excursión en **un solo sentido**, y el gel pasa
  más tiempo relajado que contraído, así que la distribución queda con una cola larga hacia ese
  lado. El ruido de medición es simétrico: `skew ≈ 0`. Este criterio **no depende de ningún
  umbral**.
- **`max_excursion_MAD`** — la mayor excursión medida en unidades de ruido. Es, literalmente, el
  SNR del canal.

El canal con mayor SNR gana. Sobre Video_prueba da `center_px` con 82 (contra 17 de
`thickness_px`); sobre Video_063, 44 contra 3 — y ese 3 es la razón por la que el detector viejo
no encontraba nada en ese vídeo.

**Ejes de la figura:** panel izquierdo — `x` = tiempo (s), `y` = el canal después de quitarle la
deriva (px); las líneas grises punteadas están a ±4×MAD. Panel derecho — histograma del mismo
canal, `x` = valor (px), `y` = número de fotogramas **en escala logarítmica** (log, porque lo
interesante son las colas, que en escala lineal no se ven).

In [ ]:
from scipy.stats import skew as _skew

CANALES = ["thickness_px", "center_px", "y_top_px", "y_bottom_px"]

filas, sin_deriva = [], {}
for c in CANALES:
    r = cr.detrend_median(df[c].to_numpy(float), fps, WIN_DERIVA_S)
    m = cr.mad(r)
    sin_deriva[c] = (r, m)
    filas.append({
        "canal"             : c,
        "recorrido_px"      : round(float(df[c].max() - df[c].min()), 3),
        "ruido_MAD_px"      : round(m, 4),
        "skew"              : round(float(_skew(r)), 2),
        "max_excursion_MAD" : round(float(np.abs(r).max() / m), 1),
        "pct_fuera_4sigma"  : round(100 * float(np.mean(np.abs(r) > 4 * m)), 2),
    })
comparacion = pd.DataFrame(filas)
display(comparacion)

CANAL_USADO = (comparacion.loc[comparacion.max_excursion_MAD.idxmax(), "canal"]
               if CANAL == "auto" else CANAL)
print(f"Canal de detección: {CANAL_USADO}"
      + ("   (elegido automáticamente por SNR)" if CANAL == "auto" else "   (forzado a mano)"))

razon = (comparacion.set_index("canal").max_excursion_MAD[CANAL_USADO]
         / comparacion.set_index("canal").max_excursion_MAD["thickness_px"])
if CANAL_USADO != "thickness_px":
    print(f"Es {razon:.1f} veces más sensible que thickness_px. El grosor no se descarta: "
          f"se mide en la sección 11, promediando eventos.")

fig, axes = plt.subplots(len(CANALES), 2, figsize=(13, 2.4 * len(CANALES)),
                         gridspec_kw={"width_ratios": [3, 1]})
for (ax1, ax2), c in zip(axes, CANALES):
    r, m = sin_deriva[c]
    ax1.plot(t, r, color="crimson" if c == CANAL_USADO else "gray", lw=0.6)
    ax1.axhline( 4 * m, color="gray", ls="--", lw=0.8)
    ax1.axhline(-4 * m, color="gray", ls="--", lw=0.8)
    ax1.set_ylabel(f"{c}\n(sin deriva, px)", fontsize=8)
    ax1.set_title(f"{c}   ruido {m:.4f} px · skew {float(_skew(r)):+.2f} · SNR "
                  f"{np.abs(r).max()/m:.0f}" + ("   <-- USADO" if c == CANAL_USADO else ""),
                  fontsize=9)
    ax1.grid(alpha=0.3)
    ax2.hist(r, bins=80, color="steelblue"); ax2.set_yscale("log")
    ax2.axvline(0, color="black", lw=0.8)
    ax2.set_ylabel("nº fotogramas (log)", fontsize=7)
axes[-1][0].set_xlabel("Tiempo (s)"); axes[-1][1].set_xlabel("valor (px)")
plt.tight_layout(); plt.show()

## 8. Línea base (estado relajado), ruido y umbral

**Entra:** el canal elegido. **Sale:** el estado relajado, el ruido medido y el umbral de amplitud.

Todo el motor de detección (`src/event_detection.py`) sigue siendo el mismo y **no se modificó**:
lo único que cambia es qué columna se le da de comer. Como ese motor busca deflexiones **hacia
abajo** y en `center_px` las contracciones van hacia arriba, se construye una columna auxiliar
`senal_contraccion` con el signo dado vuelta si hace falta. El signo se decide mirando qué cola
de la distribución es más pesada, **no suponiéndolo**: depende de la convención de la imagen y de
si el gel sube o baja al contraerse, y eso cambia entre montajes.

La línea base se estima con un **percentil móvil alto (90 %)**, apoyándose en que las
contracciones son minoritarias en el tiempo. La señal de trabajo es la *profundidad* =
base − señal.

El detalle importante: **ninguna ventana está fijada en segundos**. El ancho de evento se mide
primero en los propios datos y todas las ventanas se derivan de él. Por eso el mismo código
funciona con contracciones de 0.1 s o de 3 s sin retocar nada — y por eso este motor maneja bien
un tren rápido y uno lento en el mismo vídeo.

El ruido se mide con MAD, que a diferencia del desvío estándar no se distorsiona por los propios
eventos grandes.

**Ejes:** panel superior — `x` = tiempo (s), `y` = el canal de detección (px), con la señal cruda
en gris, el suavizado ligero en rojo y la línea base en azul punteado. Panel inferior — `x` =
tiempo (s), `y` = profundidad (px), o sea cuánto se apartó la señal de su estado relajado, con el
umbral marcado.

In [ ]:
# Signo: que los eventos queden como deflexiones HACIA ABAJO, que es lo que busca el motor.
r_sel, m_sel = sin_deriva[CANAL_USADO]
SIGNO = cr._signo_evento(r_sel)
df["senal_contraccion"] = -SIGNO * df[CANAL_USADO]
print(f"Los eventos en {CANAL_USADO} son excursiones hacia "
      f"{'ARRIBA' if SIGNO > 0 else 'ABAJO'} en coordenadas de imagen "
      f"(signo aplicado: {-SIGNO:+d})")

res = ed.detect_contractions(
    df, raw_col="senal_contraccion", amp_k=AMP_K, sharpness_threshold=SHARPNESS,
    min_amplitude_px=MIN_AMP_PX, noise_px=NOISE_PX,
)

print(f"Ancho de evento medido en los datos : {res.event_width_s:.3f} s")
print(f"Ruido {'medido' if NOISE_PX else 'estimado'} (MAD)            : {res.noise_px:.4f} px")
print(f"Umbral de amplitud (AMP_K={AMP_K:g})        : {res.amp_threshold_px:.4f} px")
print(f"Candidatos: {res.diagnostics.get('n_candidatos', 0)}"
      f" | descartados por agudeza: {res.diagnostics.get('descartados_por_agudeza', 0)}"
      f" | EVENTOS: {len(res.events)}")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a1.plot(t, df.senal_contraccion, color="lightgray", lw=0.7, label="canal de detección")
a1.plot(t, res.signal,   color="crimson",   lw=1.0, label="suavizado ligero")
a1.plot(t, res.baseline, color="steelblue", lw=1.0, ls="--", label="estado relajado (P90 móvil)")
a1.set_ylabel(f"{CANAL_USADO}, con signo\n({UNIDAD})"); a1.legend(fontsize=8)
a1.set_title(f"Señal de detección y línea base — canal {CANAL_USADO}")

a2.plot(t, res.depth, color="#145B5E", lw=0.8, label="profundidad = base − señal")
a2.axhline(res.amp_threshold_px, color="crimson", ls=":", lw=1.2,
           label=f"umbral = {AMP_K:g} × ruido = {res.amp_threshold_px:.3f}")
a2.set_xlabel("Tiempo (s)"); a2.set_ylabel(f"Profundidad ({UNIDAD})")
a2.legend(fontsize=8); a2.set_title("Señal de detección")
plt.tight_layout(); plt.show()

## 9. Detección de contracciones

**Entra:** la profundidad y el umbral. **Sale:** la tabla de eventos con sus métricas.

Superar el umbral de amplitud no alcanza: una deriva lenta de foco también lo supera. El filtro
que las separa es la **agudeza**:

$$\text{agudeza} = \frac{\text{amplitud cruda}}{\text{amplitud tras suavizado ancho}}$$

Si se suaviza con una ventana de 3× el ancho de evento medido, un transitorio real pierde casi
toda su amplitud (agudeza ≈ 1.5 o más) mientras que una deriva lenta apenas cambia (agudeza ≈ 1).
Como la ventana se deriva del ancho medido, el criterio es escala-invariante.

**Columnas de la tabla de eventos:**

| columna | qué es |
|---|---|
| `evento` | número correlativo |
| `frame_idx` | fotograma del mínimo |
| `tiempo_s` | instante del mínimo (s) |
| `amplitud_px` | cuánto se apartó del estado relajado (px del canal de detección) |
| `amplitud_pct` | lo mismo, en % de la línea base |
| `agudeza` | el cociente de arriba; < `SHARPNESS` se descarta |
| `duracion_s` | ancho a media altura del evento |
| `grosor_relajado_px` / `grosor_minimo_px` | valor de la línea base y del mínimo. **Nombres heredados**: si el canal no es `thickness_px`, son valores del canal de detección, no grosores |
| `intervalo_s` | tiempo desde el evento anterior |

In [ ]:
eventos = res.events
print(f"{len(eventos)} contracciones detectadas sobre {CANAL_USADO}")
display(eventos.head(12))

if len(eventos):
    display(eventos[["amplitud_px", "amplitud_pct", "agudeza",
                     "duracion_s", "intervalo_s"]].describe().round(3))
    display(Image(str(plotting.plot_amplitudes(eventos, res, OUT_DIR, unit_label=UNIDAD))))

## 10. Verificación: ¿son eventos reales o ruido?

**Entra:** la serie temporal. **Sale:** la curva de estabilidad y el control de falsos positivos.

Elegir `AMP_K` a dedo sería arbitrario. Se barre un rango de umbrales y se cuenta cuántos eventos
sobreviven a cada uno:

- si el conteo se mantiene en una **meseta** a lo largo de varios valores de *k*, los eventos son
  reales y el resultado no depende del umbral elegido;
- si **decae monótonamente** sin meseta, lo que se está contando es ruido.

Como control adicional se corre el mismo detector sobre la señal **invertida**: una contracción
sólo puede ir en un sentido, así que cualquier detección del lado contrario sólo puede ser ruido.

Se hacen **dos versiones del control**, y conviene mirar las dos:

- la de `event_detection` (línea base por percentil 90). Tiene un sesgo conocido y documentado:
  con eventos grandes reales, invertir la señal contra un percentil alto genera falsos positivos
  espurios, así que **sobreestima** el ruido. Sirve como alarma, no como tasa exacta.
- la de `contraction_report` (deriva quitada con mediana móvil). Es simétrica por construcción y
  da un control más limpio.

**Este paso se hace siempre antes de reportar un número de eventos.**

**Ejes de `05_estabilidad_umbral.png`:** `x` = k (el umbral en múltiplos del ruido), `y` = número
de eventos detectados con ese k. Lo que se busca es una zona horizontal.

In [ ]:
kw   = dict(raw_col="senal_contraccion", sharpness_threshold=SHARPNESS,
            min_amplitude_px=MIN_AMP_PX, noise_px=NOISE_PX)
scan = ed.threshold_stability_scan(df, **kw)
fp   = ed.symmetric_false_positive_check(df, amp_k=AMP_K, **kw)
display(scan)
print(f"Control (percentil 90): {fp['n_abajo']} a favor / {fp['n_arriba']} en contra"
      f"  (razón {fp['razon_arriba_abajo']})  ->  {fp['veredicto']}")

# Segundo control, con deriva quitada por mediana móvil (simétrico por construcción)
r_det = SIGNO * cr.detrend_median(df[CANAL_USADO].to_numpy(float), fps, WIN_DERIVA_S)
scan2 = cr.escaneo_estabilidad(r_det, fps, sep_s=max(0.3, 0.8 * res.event_width_s))
print("\nControl simétrico con mediana móvil (falsos = mismo detector sobre la señal invertida):")
display(scan2)

meseta = scan2[scan2.falsos_control == 0]
if len(meseta):
    print(f"Meseta limpia (0 falsos) desde k={meseta.k.min():g}: "
          f"{meseta.eventos.iloc[0]} eventos. Elegir AMP_K dentro de ese rango.")
else:
    print("NO hay meseta con 0 falsos: tratar el conteo con mucha cautela.")

display(Image(str(plotting.plot_threshold_stability(scan, OUT_DIR, amp_k_used=AMP_K))))

## 11. Adelgazamiento y poblaciones de eventos

**Entra:** los eventos y el canal de grosor. **Sale:** cuánto adelgaza realmente el gel, y si hay
más de un tipo de contracción en el vídeo.

### 11a. Promedio de eventos alineados

El grosor sigue siendo la variable biomecánicamente interesante: es la que se relaciona con la
deformación del material. Lo que no sirve es usarla para *detectar*. La forma de medirla cuando
está por debajo del ruido por fotograma es **alinear todos los eventos en su pico y promediarlos**:
el ruido baja como √N mientras la señal se mantiene. Con 5 eventos, el adelgazamiento de Video_063
pasó de ~1 σ por fotograma a 11 σ.

Se reportan **dos medidas**, y hay que usar la robusta:

- **`mínimo`** — el mínimo del promedio alineado. Puede estar contaminado.
- **`robusto`** — promedio de los 3 fotogramas **posteriores** al pico.

El motivo es un artefacto real: en el fotograma de máxima velocidad, el grosor medido da un salto
**positivo** (+0.39 px en Video_063). Eso no es engrosamiento, es **motion blur** — el borde se
emborrona por el movimiento y los dos bordes se "abren". El cuaderno lo detecta y avisa.

**Ejes:** `x` = tiempo respecto del pico del evento (s, 0 = pico); eje `y` izquierdo (azul) =
traslación promedio (px); eje `y` derecho (rojo) = cambio de grosor promedio (px). Los dos ejes
tienen escalas distintas a propósito: el punto es comparar la **forma**, no la magnitud.

### 11b. Poblaciones

Un mismo vídeo puede tener contracciones **espontáneas** (chicas y rápidas) y **estimuladas**
(grandes y lentas). Promediarlas juntas da una amplitud y una frecuencia que no describen a
ninguna de las dos. Si las amplitudes se separan en dos grupos (razón de medianas ≥ 2), se
reportan por separado.

In [ ]:
if len(eventos) == 0:
    print(">>> No se detectaron eventos: nada que promediar. Revisar la meseta de la sección 10.")
else:
    picos = eventos.frame_idx.to_numpy(int)

    rg = cr.detrend_median(df.thickness_px.to_numpy(float), fps, WIN_DERIVA_S)
    mg = cr.mad(rg)
    lag, prom_g, n_prom = cr.promedio_alineado(rg,    picos, fps, HALF_EVENTO_S)
    _,   prom_c, _      = cr.promedio_alineado(r_det, picos, fps, HALF_EVENTO_S)

    ruido_prom = mg / np.sqrt(max(n_prom, 1))
    i0   = int(np.argmin(np.abs(lag)))          # fotograma del pico
    imin = int(np.argmin(prom_g))
    cola = prom_g[i0 + 1: i0 + 4]
    pico_c = float(np.max(prom_c))

    print(f"Eventos promediados: {n_prom}")
    print(f"Ruido del grosor por fotograma : {mg:.4f} px")
    print(f"Ruido del promedio (MAD/sqrt N): {ruido_prom:.4f} px")
    print(f"Traslación en el pico          : {pico_c:.3f} px")
    print(f"Adelgazamiento  mínimo         : {-prom_g[imin]:.4f} px "
          f"({abs(prom_g[imin])/ruido_prom:.1f} sigma, a {lag[imin]:+.2f} s del pico)"
          f"  =  {100*abs(prom_g[imin])/pico_c:.1f} % de la traslación")
    if len(cola):
        print(f"Adelgazamiento  ROBUSTO        : {-cola.mean():.4f} px "
              f"({abs(cola.mean())/(ruido_prom/np.sqrt(len(cola))):.1f} sigma)"
              f"  =  {100*abs(cola.mean())/pico_c:.1f} % de la traslación   <-- usar éste")

    ven = prom_g[max(0, i0 - 2): i0 + 2]
    if len(ven) and ven.max() > 3 * ruido_prom:
        print(f"\nAVISO: el grosor da un salto POSITIVO de {ven.max():.3f} px "
              f"({ven.max()/ruido_prom:.1f} sigma) en el fotograma más rápido. No es "
              f"engrosamiento: es motion blur. Usar la medida robusta.")

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 3.6),
                                   gridspec_kw={"width_ratios": [1, 2]})
    axA.plot(lag, prom_c, color="steelblue", lw=1.6, label="traslación")
    axAb = axA.twinx()
    axAb.plot(lag, prom_g, color="crimson", lw=1.6, label="grosor")
    axAb.axhline(0, color="crimson", lw=0.6, ls=":")
    axA.axvline(0, color="gray", lw=0.8)
    axA.set_xlabel("t respecto del pico (s)")
    axA.set_ylabel(f"traslación ({UNIDAD})", color="steelblue")
    axAb.set_ylabel(f"grosor ({UNIDAD})", color="crimson")
    axA.set_title(f"Promedio de {n_prom} eventos alineados", fontsize=10)

    # --- poblaciones ---
    pobl = cr._poblaciones(eventos.tiempo_s.to_numpy(), eventos.amplitud_px.to_numpy())
    axB.scatter(eventos.tiempo_s, eventos.amplitud_px, s=30, color="crimson", zorder=5)
    axB.vlines(eventos.tiempo_s, 0, eventos.amplitud_px, color="steelblue", lw=1)
    axB.set_xlabel("Tiempo (s)"); axB.set_ylabel(f"Amplitud ({UNIDAD})")
    axB.set_title("Amplitud por evento", fontsize=10); axB.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    if pobl:
        print("\nDOS POBLACIONES de eventos, separadas por amplitud:")
        display(pd.DataFrame(pobl).round(3))
    else:
        print("\nUna sola población de eventos (las amplitudes no se separan en dos grupos).")

## 12. Ritmo: segmentación por frecuencia y perfil temporal

**Entra:** la lista de eventos. **Sale:** los tramos de frecuencia homogénea y el perfil de
frecuencia del vídeo.

`segment_by_rhythm` agrupa los eventos en tramos de ritmo estable. Un intervalo que es ~N veces el
ritmo local se interpreta como N−1 latidos perdidos (mismo tramo) sólo si N es chico; un salto
mayor se lee como un cambio de frecuencia de estimulación y abre un tramo nuevo.

El **perfil de frecuencia** se calcula por autocorrelación en ventana móvil, de forma
independiente de la detección de eventos: es el método más robusto para ver cambios de
estimulación, porque no depende de haber detectado cada contracción. Su límite: **no resuelve
períodos mayores a la mitad de `FREQ_WINDOW_S`** — para estimulación a 0.1 Hz (10 s de período)
hace falta `FREQ_WINDOW_S ≥ 20`.

El CV del intervalo indica la regularidad: por debajo de ~3 % es compatible con estímulo externo.

**Ejes de `04_perfil_frecuencia.png`:** `x` = tiempo (s), `y` = período dominante (s) **en escala
logarítmica**. Log porque los períodos de interés abarcan dos órdenes de magnitud (0.5 s
espontáneo, 10 s estimulado) y en escala lineal el rápido se aplastaría contra el eje.

In [ ]:
if len(eventos) == 0:
    print(">>> No se detectaron contracciones: revisar la meseta de la sección 10.")
else:
    if FREQ_WINDOW_S < 2 * eventos.intervalo_s.median(skipna=True):
        print(f"AVISO: FREQ_WINDOW_S={FREQ_WINDOW_S:g}s no resuelve períodos mayores a "
              f"{FREQ_WINDOW_S/2:g}s, y el intervalo mediano medido es "
              f"{eventos.intervalo_s.median(skipna=True):.2f}s. Subir FREQ_WINDOW_S.")

    ev   = ed.segment_by_rhythm(eventos)
    seg  = ed.analyze_segments(ev, res.signal, t)
    freq = ed.frequency_profile(res.signal, t, window_s=FREQ_WINDOW_S)

    display(seg)
    for _, s in seg.iterrows():
        print(f"Tramo {int(s.segmento)}: {int(s.n_eventos)} eventos | "
              f"t={s.t_inicio_s:.1f}–{s.t_fin_s:.1f}s | {s.frecuencia_Hz:.3f} Hz "
              f"(T={s.periodo_s:.3f}s) | amp={s.amplitud_media_px:.3f} px | "
              f"CV={s.CV_intervalo_pct:.1f}% ({ed.interpret_regularity(s.CV_intervalo_pct)})")

    display(Image(str(plotting.plot_events(df, res, ev, OUT_DIR, unit_label=UNIDAD,
                                           raw_col="senal_contraccion"))))
    display(Image(str(plotting.plot_frequency_profile(freq, OUT_DIR, window_s=FREQ_WINDOW_S))))

    ventanas = plotting.auto_windows_from_segments(seg)
    display(Image(str(plotting.plot_segment_comparison(df, res, ev, OUT_DIR, ventanas,
                                                       unit_label=UNIDAD,
                                                       raw_col="senal_contraccion"))))

## 13. Exportación de resultados

**Entra:** todo lo anterior. **Sale:** los archivos que quedan en
`data/processed_data/<nombre_del_vídeo>/`.

| archivo | contenido |
|---|---|
| `serie_temporal.xlsx` | hoja `diagnostics`: una fila por fotograma con los 4 canales, residuos, `outlier_frac` y etiqueta de calidad. Hoja `resumen`: parámetros usados y estadísticos globales |
| `eventos.xlsx` | hojas `diagnostics` (los eventos), `resumen`, `por_segmento`, `perfil_frecuencia` y `estabilidad_umbral` |
| `contracciones.xlsx` | el reporte de `contraction_report.py`: `estab_*`, `resumen_*`, `eventos_*` y `poblac_*` |
| `00_max_projection.png` | el mapa de máxima intensidad |
| `00_roi_profile.png` | perfil de grosor y nitidez con la ROI marcada |
| `01_serie_temporal.png` … `06_comparacion_tramos.png` | las figuras de arriba |

Son exactamente los archivos que producen `main.py` y `scripts/contraction_report.py`; este
cuaderno no genera un formato paralelo.

In [ ]:
from src.qc_visualization import _write_xlsx

serie_xlsx = _write_xlsx(df, OUT_DIR / "serie_temporal.xlsx", summary={
    "video"                 : VIDEO.name,
    "fotogramas"            : len(df),
    "px_to_mm"              : PX_TO_MM,
    "canal_de_deteccion"    : CANAL_USADO,
    "ROI x_start"           : roi["x_start"],
    "ROI x_end"             : roi["x_end"],
    "ROI metodo"            : q.get("method"),
    "ransac_degree"         : CFG.ransac_degree,
    "ransac_umbral"         : CFG.ransac_residual_threshold or "adaptativo",
    "use_clahe"             : CFG.use_clahe,
    "grosor medio (px)"     : round(float(df.thickness_px.mean()), 3),
    "recorrido centro (px)" : round(float(df.center_px.max() - df.center_px.min()), 3),
})

if len(eventos):
    eventos_xlsx = OUT_DIR / "eventos.xlsx"
    _write_xlsx(ev, eventos_xlsx, summary={
        "archivo_fuente"        : str(VIDEO),
        "canal_de_deteccion"    : CANAL_USADO,
        "ancho_evento_medido_s" : round(res.event_width_s, 4),
        "ruido_px"              : round(res.noise_px, 5),
        "ruido_es_provisorio"   : NOISE_PX is None,
        "umbral_amplitud_px"    : round(res.amp_threshold_px, 5),
        "amp_k"                 : AMP_K,
        "umbral_agudeza"        : SHARPNESS,
        "n_eventos"             : len(ev),
        "n_segmentos"           : int(ev.segmento.nunique()),
        "control_FP_abajo"      : fp["n_abajo"],
        "control_FP_arriba"     : fp["n_arriba"],
        "control_FP_veredicto"  : fp["veredicto"],
    })
    ac._append_sheet(eventos_xlsx, seg,   "por_segmento")
    ac._append_sheet(eventos_xlsx, freq,  "perfil_frecuencia")
    ac._append_sheet(eventos_xlsx, scan,  "estabilidad_umbral")
    ac._append_sheet(eventos_xlsx, scan2, "estabilidad_simetrica")
    ac._append_sheet(eventos_xlsx, comparacion, "comparacion_canales")

print("Archivos generados en", OUT_DIR.relative_to(PROJECT_ROOT))
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<28} {f.stat().st_size/1024:8.1f} kB")

---

## Correspondencia con MATLAB (`Contraction_Analysis.mlx`) y con MuscleMotion

El live script de Tecnun parte de un `.txt` de dos columnas (tiempo, amplitud) que ya existe.
Este cuaderno cubre ese mismo tramo **y además el que lo precede**: la construcción de esa señal a
partir del vídeo.

| `Contraction_Analysis.mlx` | Este cuaderno |
|---|---|
| 1. Importación de `contraction.txt` y recorte de artefactos | 2–6. El vídeo se convierte en la señal, midiendo los bordes fotograma a fotograma |
| 1. `detrend` para centrar la señal | 8. Percentil móvil al 90 %: sigue la deriva lenta en lugar de suponerla lineal |
| 2. `findpeaks` con `MinPeakDistance` fijo | 9. `find_peaks` con distancia derivada del ancho de evento medido en los datos |
| 2. Basal por mediana móvil (`Window_Size` fijo) | 8. Basal por percentil móvil, con ventana derivada del ancho de evento |
| 3. Inicio y fin de cada pico por cruce con la basal | 9. Duración a media altura sobre la señal de profundidad |
| 4. Métricas por pico y exportación a Excel | 9 y 12. Amplitud, agudeza, duración, intervalo, y por tramo frecuencia, CV y latidos perdidos |
| — | 7. **Elección del observable con datos** |
| — | 10. Verificación estadística del umbral (meseta + dos controles de falsos positivos) |
| — | 11. Adelgazamiento por promedio de eventos alineados, y separación en poblaciones |
| — | 3–5. Control de calidad visual del borde |

### Frente a MuscleMotion

MuscleMotion y las herramientas basadas en flujo óptico o diferencia de píxeles miden **variación
de intensidad** en el tiempo. Un cambio sutil de iluminación —una sombra que se mueve, un LED que
parpadea, el foco que se reajusta— produce exactamente la misma clase de señal que una
contracción: misma unidad, misma escala.

Este flujo mide la **posición geométrica de un borde**, no su brillo. Un cambio de iluminación
puede correr el borde detectado unas décimas de píxel si altera el gradiente, pero no genera un
desplazamiento de varios píxeles con la forma temporal de una contracción.

Esa ventaja hoy está **argumentada, no medida**. Lo que falta para convertirla en un número es un
vídeo de control donde **sólo cambie la luz y el gel esté quieto**: mismo campo, mismo gel,
inmóvil, con un cambio de iluminación gradual o un parpadeo. Sobre ese vídeo, este flujo debería
dar una serie plana. Es barato de grabar y es lo que convence en un artículo.